In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

In [ ]:
ds = pd.read_csv('data/enriched_dataset.csv', index_col = 0)
ds

,ticker,year,Выручка,Операционная прибыль,EBITDA,Чистая прибыль,Опер.денежный поток,CAPEX,FCF,"Див доход, ао",...,CAPEX/Выручка,total_price,total_price_target,revenue,total_price_momentum,momentum,lin_reg_pred,log_reg_pred,reports,llm_pred
0,ENPG,2022,1134.0,137.50,213.80,74.20,39.20,117.300,-39.10,0.0,...,10.0,374.000,440.200,0.162973,885.000,-0.861332,0.255462,0.479567,NaN,продавать
1,ENPG,2023,1249.0,87.80,183.90,50.80,232.00,123.400,58.00,0.0,...,10.0,440.200,347.300,-0.237040,374.000,0.162973,-0.075867,0.430704,NaN,держать
2,ENPG,2024,1335.0,137.20,266.70,90.80,151.10,171.100,-91.40,0.0,...,13.0,347.300,454.400,0.268789,440.200,-0.237040,0.101972,0.477232,NaN,покупать
3,HEAD,2022,18.1,6.92,9.16,6.01,7.67,0.450,7.00,0.0,...,2.0,1211.000,2951.000,0.890698,3899.408,-1.169378,0.380008,0.545556,NaN,покупать
4,HEAD,2023,29.4,15.80,17.40,12.50,16.40,0.517,16.40,0.0,...,2.0,2951.000,4604.010,0.444784,1211.000,0.890698,-0.272155,0.482701,NaN,покупать
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439,YKEN,2022,25.3,12.10,5.30,8.00,0.67,5.950,-8.51,0.0,...,23.0,0.264,0.609,0.835869,0.305,-0.144363,0.031411,0.405846,NaN,держать
440,YKEN,2023,32.2,7.20,5.90,3.00,5.12,10.300,-9.47,0.0,...,32.0,0.609,0.480,-0.238032,0.264,0.835869,-0.438991,0.347454,NaN,продавать
441,YDEX,2022,521.7,13.20,64.10,10.80,41.70,50.500,-10.90,0.0,...,10.0,1814.000,2542.000,0.337417,4502.000,-0.908987,0.267577,0.450654,['There were also employee share options outst...,держать
442,YDEX,2023,798.1,63.90,120.80,52.10,115.60,84.200,-37.70,0.0,...,11.0,2542.000,4137.120,0.487049,1814.000,0.337417,-0.163204,0.423087,NaN,держать


In [4]:
true_long = ds.groupby('year')['revenue'].rank(pct=True) >= 0.8
true_short = ds.groupby('year')['revenue'].rank(pct=True) <= 0.2

In [7]:
# Метрики на train датасете

llm_long = ds[ds["year"] <= 2023]["llm_pred"] == "покупать"
llm_short = ds[ds["year"] <= 2023]["llm_pred"] == "продавать"

In [8]:
print("LLM long confusion matrix")
cm = confusion_matrix(
    llm_long.astype(np.int32),
    true_long[ds["year"] <= 2023].astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("LLM short confusion matrix")
cm = confusion_matrix(
    llm_short.astype(np.int32),
    true_short[ds["year"] <= 2023].astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

LLM long confusion matrix
[[240  29]
 [ 50  47]]
Precision: 0.4845360824742268, Recall: 0.618421052631579, F1: 0.5433526011560693


LLM short confusion matrix
[[243  29]
 [ 50  44]]
Precision: 0.46808510638297873, Recall: 0.6027397260273972, F1: 0.5269461077844312




In [ ]:
# Доходность на train датасете
llm_revenue = (
    (ds[ds["year"] <= 2023][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
    + (1 - ds[ds["year"] <= 2023][ds["llm_pred"] == "продавать"].groupby('year')["revenue"].mean()) # short
)
print(f"LLM revenue: {llm_revenue}") # видно что немного оверфитнулись под трейн, ну это и логично, GRPO же

oracle_revenue = (
    (ds[ds["year"] <= 2023][true_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - ds[ds["year"] <= 2023][true_short == 1].groupby('year')["revenue"].mean()) # short
)
print(f"Oracle revenue: {oracle_revenue}")

LLM revenue: year
2021    0.548290
2022    1.106600
2023    0.747051
Name: revenue, dtype: float64
Oracle revenue: year
2021    0.937214
2022    1.702441
2023    1.223855
Name: revenue, dtype: float64


/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/3262938707.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  (ds[ds["year"] <= 2023][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/3262938707.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  + (1 - ds[ds["year"] <= 2023][ds["llm_pred"] == "продавать"].groupby('year')["revenue"].mean()) # short
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/3262938707.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  (ds[ds["year"] <= 2023][true_long == 1].groupby('year')["revenue"].mean() - 1) # long
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/3262938707.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  + (1 - ds[ds["year"] <= 2023][true_short == 1].groupby('year')["reven

In [13]:
# Метрики на test датасете

test_llm_long = ds[ds["year"] >= 2024]["llm_pred"] == "покупать"
test_llm_short = ds[ds["year"] >= 2024]["llm_pred"] == "продавать"

In [15]:
print("LLM long confusion matrix")
cm = confusion_matrix(
    test_llm_long.astype(np.int32),
    true_long[ds["year"] >= 2024].astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("LLM short confusion matrix")
cm = confusion_matrix(
    test_llm_short.astype(np.int32),
    true_short[ds["year"] >= 2024].astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

LLM long confusion matrix
[[52  5]
 [10 11]]
Precision: 0.5238095238095238, Recall: 0.6875, F1: 0.5945945945945946


LLM short confusion matrix
[[51  7]
 [12  8]]
Precision: 0.4, Recall: 0.5333333333333333, F1: 0.45714285714285713




In [16]:
# Доходность на test датасете
test_llm_revenue = (
    (ds[ds["year"] >= 2024][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
    + (1 - ds[ds["year"] >= 2024][ds["llm_pred"] == "продавать"].groupby('year')["revenue"].mean()) # short
)
print(f"LLM revenue: {test_llm_revenue}")

test_oracle_revenue = (
    (ds[ds["year"] >= 2024][true_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - ds[ds["year"] >= 2024][true_short == 1].groupby('year')["revenue"].mean()) # short
)
print(f"Oracle revenue: {test_oracle_revenue}") 

LLM revenue: year
2024    0.404919
Name: revenue, dtype: float64
Oracle revenue: year
2024    0.81648
Name: revenue, dtype: float64


/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2236475471.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  (ds[ds["year"] >= 2024][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2236475471.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  + (1 - ds[ds["year"] >= 2024][ds["llm_pred"] == "продавать"].groupby('year')["revenue"].mean()) # short
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2236475471.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  (ds[ds["year"] >= 2024][true_long == 1].groupby('year')["revenue"].mean() - 1) # long
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2236475471.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  + (1 - ds[ds["year"] >= 2024][true_short == 1].groupby('year')["reven

In [ ]:
# Посмотрим насколько полезными оказались сниппеты из мсфо-отчестности

llm_revenue_with_reports = (
    (ds[(ds["year"] <= 2023) & (~ds["reports"].isna())][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
    + (1 - ds[(ds["year"] <= 2023) & (~ds["reports"].isna())][ds["llm_pred"] == "продавать"].groupby('year')["revenue"].mean()) # short
)
print(f"LLM revenue with reports: {llm_revenue_with_reports}")
print()

llm_revenue_without_reports = (
    (ds[(ds["year"] <= 2023) & (ds["reports"].isna())][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
    + (1 - ds[(ds["year"] <= 2023) & (ds["reports"].isna())][ds["llm_pred"] == "продавать"].groupby('year')["revenue"].mean()) # short
)
print(f"LLM revenue witout reports: {llm_revenue_without_reports}")

LLM revenue with reports: year
2021    0.689829
2022    1.161774
2023    0.599873
Name: revenue, dtype: float64

LLM revenue witout reports: year
2021    0.464304
2022    0.972025
2023    0.867130
Name: revenue, dtype: float64


/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2244894322.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  (ds[(ds["year"] <= 2023) & (~ds["reports"].isna())][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2244894322.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  + (1 - ds[(ds["year"] <= 2023) & (~ds["reports"].isna())][ds["llm_pred"] == "продавать"].groupby('year')["revenue"].mean()) # short
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2244894322.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  (ds[(ds["year"] <= 2023) & (ds["reports"].isna())][ds["llm_pred"] == "покупать"].groupby('year')["revenue"].mean() - 1) # long
/var/folders/fn/q7t5q9rd3_lb1b2cypj6f3h00000gn/T/ipykernel_59587/2244894322.py:12: UserWarning: Boolean Series key will be reindexed